

# **Laboratorio 10: Chatbot 101 💡**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2026</strong></center>

### Cuerpo Docente:

- Profesores: Pablo Badilla, Diego Cortez
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes

### **Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados**

- Nombre de alumno 1: Javier Cruz Araneda
- Nombre de alumno 2: Enzo Toledo Venegas

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/enzo-toledo/MDS7202)

## **Temas a tratar**

- Large Language Models
- Output parsers
- Chatbot con RAG
- Memoria
- Análisis de embeddings

### **Objetivos principales del laboratorio**

- Resolución de problemas secuenciales usando Reinforcement Learning
- Habilitar un Chatbot para entregar respuestas útiles usando Large Language Models.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

### **0 Configuración Inicial**

<p align="center">
  <img src="https://media1.tenor.com/m/uqAs9atZH58AAAAd/config-config-issue.gif"
" width="400">
</p>

Como siempre, cargamos todas nuestras API KEY al entorno:

In [1]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

### **1. Retrieval Augmented Generation (1.0 puntos)**

#### **1.1 Reunir Documentos (0.1 puntos)**

Reuna documentos PDF sobre los que hacer preguntas siguiendo las siguientes instrucciones:
  - 2 documentos .pdf como mínimo, 5 como máximo.
  - 30 páginas de contenido como mínimo entre todos los documentos.
  - Ideas para documentos: Documentos relacionados a temas académicos, laborales o de ocio. Aprovechen este ejercicio para construir algo útil y/o relevante para ustedes!
  - Deben ocupar documentos reales, no pueden utilizar los mismos de la clase.
  - Deben registrar sus documentos en la siguiente [planilla](https://docs.google.com/spreadsheets/d/1fv7WV273_rjoFS0ORvnn-HkFYX7TCe0SNcWewwL4lkI/edit?usp=sharing). **NO PUEDEN USAR LOS MISMOS DOCUMENTOS QUE OTRO GRUPO**
  - **Recuerden adjuntar los documentos en su entrega**.

In [2]:
!uv add pyPDF2

Resolved 215 packages in 1ms
Checked 206 packages in 27ms


In [3]:
import PyPDF2

# Links de los documentos de ser necesario:
# Idea que los conecta: "Operadores Neuronales para resolver Ecuaciones en Derivadas Parciales"
# DeepONet.pdf : https://arxiv.org/pdf/1910.03193
# FNO.pdf      : https://arxiv.org/pdf/2010.08895
# PINNs.pdf    : https://arxiv.org/pdf/1711.10561

# Colocar los 3 papers en la carpeta labs/lab_10/papers/ con los nombres anteriores:
doc_paths = ["papers/DeepONet.pdf", "papers/FNO.pdf", "papers/PINNs.pdf"]

assert len(doc_paths) >= 2, "Deben adjuntar un mínimo de 2 documentos"
assert len(doc_paths) <= 5, "Deben adjuntar un máximo de 5 documentos"

total_paginas = sum(len(PyPDF2.PdfReader(open(doc, "rb")).pages) for doc in doc_paths)
assert total_paginas >= 30, f"Páginas insuficientes: {total_paginas}"

#### **1.2 Vectorizar Documentos (0.2 puntos)**

Vectorice los documentos y almacene sus representaciones de manera acorde.

In [4]:
# Librerias necesarias (considerando clases 24 y 25):
!uv add langchain_community pypdf cryptography faiss-cpu ipywidgets langchain-google-genai

Resolved 215 packages in 1ms
Checked 206 packages in 19ms


In [5]:
import asyncio

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm.notebook import tqdm

C:\Users\migue\AppData\Local\Temp\ipykernel_2972\1462583881.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


> **IMPORTANTE:** Ejecutar lo de abajo solo si no se tiene la carpeta `faiss_index`, para no consumir la key gratuita por si acaso hay que volver a cambiar documentos, agregarlos o algo así.

In [ ]:
# Cargar 3 pdfs:
docs = []
for path in doc_paths:
    docs.extend(PyPDFLoader(path).load())
print(f"Número de páginas: {len(docs)}")

# Split: dividir en chunks usando parámetros de clase:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f"Chunks generados: {len(splits)}")

# Embed y Store con pausa para no consumir toda la api gratuita:
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")


# Analogo a clase: (la pausa es para no consumir toda la API gratuita al procesar muchos documentos)
async def crear_faiss_con_pausa(documentos, embeddings):
    db = FAISS.from_documents([documentos[0]], embeddings)
    for doc in tqdm(documentos[1:], desc="Indexando documentos"):
        await asyncio.sleep(1)
        db.add_documents([doc])
    return db


vectorstore = await crear_faiss_con_pausa(splits, embedding)
vectorstore

Número de páginas: 60
Chunks generados: 128


Indexando documentos:   0%|          | 0/127 [00:00<?, ?it/s]

> **Observación:** 60 páginas consistente con lo comprobado manualmente, son 128 chunks generados y tarda aproximadamente 1.35s/it, tiempo total de ejecución de 3 minutos aprox.

In [ ]:
# Para no perder los datos procesados lo guardamos en local!
vectorstore.save_local("faiss_index")

#### **1.3 Habilitar RAG (0.4 puntos)**

Habilite la solución RAG a través de una **clase** que tenga un **método** `chat` que reciba la pregunta y un argumento opcional de n_results y retorne la respuesta con RAG. El resto de los argumentos debe recibirlos en la inicialización. Requisitos:
- La clase debe ser independiente, es decir no debe depender de objetos definidos fuera de ella. Todos los objetos deben recibirse como argumentos o ser generados por métodos de la clase. La  excepción son clases.
- **Requisito estricto:** el modelo generativo debe tener una temperatura de 1.0.

Luego instancie su clase y utilice el método `chat` con una pregunta de prueba. 

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

In [9]:
class RAG:
    def __init__(
        self,
        faiss_index_name: str,
        rag_template: str,
        chat_model_name: str = "gemini-3.1-flash-lite",
        embedding_model_name: str = "gemini-embedding-001",
    ):
        # Modelo de embeddings (mismo usado al crear el índice):
        self.embedding = GoogleGenerativeAIEmbeddings(model=embedding_model_name)

        # Cargar la vectorstore desde disco:
        self.vectorstore = FAISS.load_local(
            faiss_index_name,
            self.embedding,  # El embedding usado al crear el indice
            allow_dangerous_deserialization=True,
        )

        # LLM con temperatura 1.0 según enunciado:
        self.llm = ChatGoogleGenerativeAI(model=chat_model_name, temperature=1.0)

        # Prompt de RAG:
        self.rag_prompt = PromptTemplate.from_template(rag_template)

    # Análogo a clase 25:
    def _format_docs(self, docs):
        return "\n\n".join(f"# Contexto #{i + 1}:\n{doc.page_content}" for i, doc in enumerate(docs))

    def chat(self, question, n_results=5):
        retriever = self.vectorstore.as_retriever(
            search_type="similarity",  # el método de búsqueda
            search_kwargs={"k": n_results},  # k es el número de documentos a recuperar.
        )
        retriever_chain = retriever | self._format_docs  # la chain :)

        rag_chain = (
            {"context": retriever_chain, "question": RunnablePassthrough()}  # question pasa directo hacia el prompt
            | self.rag_prompt  # prompt con las variables question y context.
            | self.llm  # llm recibe el prompt y responde.
            | StrOutputParser()  # con esto solo se recupera la respuesta (texto).
        )
        return rag_chain.invoke(question)

In [10]:
rag_template = """
Eres un asistente experto en fundamentos matemáticos del machine learning,
con foco en redes neuronales como sistemas dinámicos, métodos numéricos para
ecuaciones diferenciales y aprendizaje profundo informado por la física.
Tu único rol es contestar preguntas del usuario a partir de la información relevante
que te sea proporcionada. Responde de la forma más completa posible usando toda la
información entregada. Responde sólo lo que te pregunten a partir de la información
relevante. NUNCA inventes una respuesta.

Información relevante: {context}
Pregunta: {question}
Respuesta útil:
"""

rag = RAG(faiss_index_name="faiss_index", rag_template=rag_template)

# Pregunta de prueba
print(rag.chat("¿Qué es una Physics-Informed Neural Network (PINN)?"))

Una Physics-Informed Neural Network (PINN) es una nueva clase de aproximador universal de funciones basada en redes neuronales profundas, diseñada para resolver tareas de aprendizaje supervisado mientras respeta leyes físicas.

Sus características principales son:

*   **Integración de leyes físicas:** Estas redes están entrenadas para codificar las leyes de la física —descritas por ecuaciones en derivadas parciales (EDP) generales, no lineales y dependientes del tiempo— como información a priori. Están restringidas para cumplir con cualquier principio de simetría, invarianza o conservación que gobierne los datos observados.
*   **Capacidad de representación:** A diferencia de otros métodos, como la regresión por procesos gaussianos, las PINNs no requieren suposiciones previas limitantes, linealización o pasos de tiempo locales, aprovechando la capacidad de las redes neuronales como aproximadores universales de funciones para abordar problemas no lineales directamente.
*   **Diferencia

#### **1.4 Verificación de respuestas (0.2 puntos)**

Genere un listado de 3 tuplas ("pregunta", "respuesta correcta") y analice la respuesta de su solución para cada una. ¿Su solución RAG entrega las respuestas que esperaba?

Ejemplo de tupla:
- Pregunta: ¿Quién es el presidente de Chile?
- Respuesta correcta: El presidente de Chile es Gabriel Boric

> **Pregunta 1:** ¿Cómo incorporan los PINNs las leyes físicas en el entrenamiento de la red? **Respuesta:** Añaden el residual de la ecuación diferencial como un término en la función de pérdida, de modo que la red se penaliza si no satisface la PDE.

In [11]:
# Pregunta 1:
print(rag.chat("¿Cómo incorporan los PINNs las leyes físicas en el entrenamiento de la red?"))

Los PINNs (Physics Informed Neural Networks) incorporan las leyes físicas en el entrenamiento de la red al ser construidos como aproximadores universales de funciones capaces de codificar cualquier ley física subyacente que gobierne un conjunto de datos, las cuales son descritas mediante ecuaciones diferenciales parciales (PDE) generales, lineales o no lineales.

El proceso se basa en los siguientes fundamentos:

*   **Restricción mediante leyes físicas:** Las redes neuronales son constreñidas para respetar principios de simetría, invarianza o leyes de conservación que originan los datos observados, modelados por las PDE que rigen el sistema.
*   **Uso de diferenciación automática:** Se aprovecha la diferenciación automática para derivar la red neuronal respecto a sus coordenadas de entrada y parámetros del modelo. Esto permite que la red sea totalmente diferenciable con respecto a todas sus entradas y parámetros libres.
*   **Codificación como información previa:** A diferencia de los

> **Pregunta 2:** ¿Cuál es el fundamento teórico de DeepONet y cómo se refleja en su arquitectura? **Respuesta:** Se basa en el teorema de aproximación universal para operadores: las redes pueden aproximar operadores, no solo funciones. Por eso usa dos sub-redes: la branch (codifica la función de entrada en puntos fijos) y la trunk (codifica las coordenadas de salida); su producto interno da el resultado del operador.

In [12]:
# Pregunta 2:
print(rag.chat("¿Cuál es el fundamento teórico de DeepONet y cómo se refleja en su arquitectura?"))

El fundamento teórico de DeepONet se basa en el **Teorema de Aproximación Universal para Operadores**, formulado por Chen & Chen. Este teorema establece que, para un operador no lineal continuo $G$ que mapea un espacio de Banach a otro, existe una representación capaz de aproximar dicho operador con un error arbitrariamente pequeño ($\epsilon$) utilizando una estructura de red específica.

Esta base teórica se refleja en la arquitectura de DeepONet de la siguiente manera:

*   **Estructura de dos ramas (Trunk-Branch):** La arquitectura separa el procesamiento de las entradas en dos sub-redes especializadas, tratando los componentes de forma distinta, como sugiere el teorema:
    *   **Red de rama (Branch network):** Se encarga de procesar la información de la función de entrada $u$ evaluada en un conjunto finito de puntos sensores $\{x_1, x_2, \dots, x_m\}$.
    *   **Red de tronco (Trunk network):** Se encarga de procesar la variable $y$ (donde se evalúa el operador), aprendiendo las 

> **Pregunta 2:** ¿Qué ventaja ofrece el FNO frente a los solvers numéricos tradicionales para resolver EDPs? **Respuesta:** El FNO aprende el operador de solución de toda una familia de EDPs, no una sola instancia. Una vez entrenado, resuelve un caso nuevo con un único forward pass, órdenes de magnitud más rápido que un solver clásico que debe iterar desde cero en cada problema. Además es independiente de la malla (entrena y evalúa en distintas resoluciones) y no requiere conocer la forma explícita de la ecuación, solo datos de entrada-salida.

In [13]:
# Pregunta 3:
print(rag.chat("¿Qué ventaja ofrece el FNO frente a los solvers numéricos tradicionales para resolver EDPs?"))

La ventaja fundamental del **Fourier Neural Operator (FNO)** frente a los solvers numéricos tradicionales (como FEM o FDM) radica en su **eficiencia computacional y capacidad de generalización**:

1.  **Velocidad superior:** El FNO es hasta tres órdenes de magnitud más rápido que los solvers tradicionales. Mientras que un solver tradicional puede tardar horas en resolver un sistema complejo, el FNO puede evaluar una instancia en una fracción de segundo (por ejemplo, 0.005s frente a 2.2s en la ecuación de Navier-Stokes). Esto permite reducir tiempos de ejecución de 18 horas a 2.5 minutos en procesos como el MCMC.
2.  **Aprendizaje de familias de ecuaciones:** A diferencia de los métodos clásicos que resuelven una instancia específica de la ecuación, los operadores neuronales aprenden el mapeo entre espacios de funciones, permitiendo resolver toda una familia de EDPs una vez entrenados.
3.  **Independencia de la resolución:** El error del FNO es invariante ante cambios en la resolución d

> **Observación:** Notemos que todas las preguntas las respondió muy bien y además incluyo información numérica propia del paper correspondiente.

#### **1.5 Persistencia de base de conocimiento (0.1 puntos)**

Guarde su base de conocimiento para reutilizarla más adelante. Su entrega deberá venir con su base de conocimiento precomputada

In [16]:
# La base ya se guardó con save_local antes. Aquí está la verificación:
vectorstore_reload = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True,
)
print(f"Vectores almacenados: {vectorstore_reload.index.ntotal}")

Vectores almacenados: 128


### **2. Creando un chatbot con RAG (2.0 puntos)**

#### **2.1 Análisis de sentimiento (0.3 puntos)**

Genere una chain que reciba una pregunta del usuario y lo clasifique según sentimiento:
- Positivo
- Neutro
- Negativo

La chain deberá retornar ESTRICTAMENTE uno de esos 3 sentimientos. Evalúe la chain con 3 ejemplos, uno para cada sentimiento.

In [19]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

In [22]:
# Modelo base para la chain de sentimiento:
llm_sentimiento = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=1.0)


# Estructura de clase 24:
class AnalisisSentimiento(BaseModel):
    sentimiento: Literal["positivo", "neutro", "negativo"] = Field(
        description="El sentimiento predominante del mensaje del usuario."
    )


sentiment_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Eres un clasificador de sentimiento. Clasifica el mensaje del usuario "
            "en una de las tres categorías permitidas según su tono emocional.",
        ),
        ("human", "{question}"),
    ]
)

# structured output garantiza que la salida sea ESTRICTAMENTE uno de los 3 sentimientos:
sentiment_chain = sentiment_prompt | llm_sentimiento.with_structured_output(AnalisisSentimiento)

# Ejemplos:
ejemplos_sentimiento = [
    "No me parece bien de tu parte hacer eso.",  # negativo
    "Gracias por tu respuesta!",  # positivo
    "Cómo funciona una red neuronal?",  # neutro
]

for texto in ejemplos_sentimiento:
    resultado = sentiment_chain.invoke({"question": texto})
    print(f"'{texto[:45]}' : {resultado.sentimiento}")

'No me parece bien de tu parte hacer eso.' : negativo
'Gracias por tu respuesta!' : positivo
'Cómo funciona una red neuronal?' : neutro


> **Observación:** Funciona :D

#### **2.2 Rag con historial de chat (1.2 puntos)**

Modifique su clase (con otro nombre) que implementa RAG de forma que para generar utilice **una lista de mensajes** anteriores de la conversación. La respuesta debe considerar la conversación completa y deben haber roles claros separados en los prompts (considere la información del siguiente [enlace](https://docs.langchain.com/oss/python/langchain/messages)). Además, debe cumplir los siguientes requisitos:
- Su función de inicialización **debe** recibir los argumentos del ejemplo presente en la celda de código, con los tipos ahí presentes
- El método `chat` NO DEBE recibir la lista de mensajes. Sólo debe recibir la pregunta del usuario, n_results (opcional) y retornar la respuesta
- Debe almacenar acumulativamente tanto el **sentimiento** detectado como el **embedding** del mensaje del usuario
- Al igual que la clase que implementa RAG, debe ser independiente y el modelo debe tener una **temperatura de 1.0**
- Considere que este es un chat con memoria, por lo que deberá poder responder correctamente interacciones que no necesariamente están en el último mensaje

In [24]:
from typing import Literal

from langchain_community.vectorstores import FAISS
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from pydantic import BaseModel, Field

In [32]:
class AnalisisSentimiento(BaseModel):
    sentimiento: Literal["positivo", "neutro", "negativo"] = Field(
        description="El sentimiento predominante del mensaje del usuario."
    )


class Chatbot:
    def __init__(
        self,
        prompts: dict,
        faiss_index_name: str,
        chat_model_name: str = "gemini-3.1-flash-lite",
        embedding_model_name: str = "gemini-embedding-001",
    ):
        self.prompts = prompts

        # Generar embeddings:
        self.embedding = GoogleGenerativeAIEmbeddings(model=embedding_model_name)

        # Carga de la base de conocimiento local generada antes (FAISS.load_local)
        self.vectorstore = FAISS.load_local(
            faiss_index_name,
            self.embedding,
            allow_dangerous_deserialization=True,
        )

        # LLM principal con temperatura 1.0 según enunciado:
        self.llm = ChatGoogleGenerativeAI(model=chat_model_name, temperature=1.0)

        # Chain de sentimiento (structured output)
        sentiment_prompt = ChatPromptTemplate.from_messages(
            [
                ("system", self.prompts["sentimiento"]),
                ("human", "{question}"),
            ]
        )
        self.sentiment_chain = sentiment_prompt | self.llm.with_structured_output(AnalisisSentimiento)

        # Estado acumulativo
        self.chat_history = []  # lista de HumanMessage / AIMessage
        self.sentimientos = []  # sentimiento de cada mensaje del usuario
        self.embeddings_usuario = []  # embedding de cada mensaje del usuario

    # igual que antes:
    def _format_docs(self, docs):
        return "\n\n".join(f"# Contexto #{i + 1}:\n{doc.page_content}" for i, doc in enumerate(docs))

    # Aquí se integra lo nuevo:
    def chat(self, question, n_results=5):
        # Almacenar acumulativamente sentimiento y embedding del mensaje del usuario
        sentimiento = self.sentiment_chain.invoke({"question": question}).sentimiento
        self.sentimientos.append(sentimiento)
        self.embeddings_usuario.append(
            self.embedding.embed_query(question)
        )  # .embed_query pues solo es un mensaje del usuario, no un documento.

        # Retrieval sobre la base de conocimiento
        retriever = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": n_results},
        )
        contexto = self._format_docs(retriever.invoke(question))

        # Prompt con roles, historial y pregunta actual:
        prompt_template = ChatPromptTemplate.from_messages(
            [
                ("system", self.prompts["sistema"]),
                MessagesPlaceholder(variable_name="historial"),
                ("human", "{question}"),
            ]
        )
        chain = prompt_template | self.llm | StrOutputParser()  # armar cadena final

        # La respuesta considera la conversación completa, ie, viendo el historial
        response = chain.invoke(
            {
                "context": contexto,
                "historial": self.chat_history,
                "question": question,
            }
        )

        # Actualizar historial con ambos roles
        self.chat_history.append(HumanMessage(content=question))
        self.chat_history.append(AIMessage(content=response))

        return response

In [33]:
prompts = {
    "sistema": """
Eres un asistente experto en fundamentos matemáticos del machine learning:
redes neuronales como sistemas dinámicos, métodos numéricos para ecuaciones
diferenciales y aprendizaje profundo informado por la física.
Responde a partir de la información relevante entregada y de la conversación previa.
Responde de forma completa y NUNCA inventes una respuesta que no se desprenda del contexto
o tu conocimiento previo. Si no tienes suficiente información hazlo saber al usuario.

Información relevante: {context}
""",
    "sentimiento": "Eres un clasificador de sentimiento. Clasifica el mensaje del "
    "usuario en una de las tres categorías permitidas según su tono.",
}

#### **2.4 Verificación de funcionalidades de chatbot (0.5 puntos)**

Instancie e inicialice su chatbot. Luego interactúe con él con 5 a 10 mensajes donde se vean diferentes sentimientos. Cada mensaje debe llamarse en una nueva celda, donde se muestre también la respuesta del chatbot. Luego de los 10 mensajes, muestre los sentimientos detectados y el historial de chat luego de toda la interacción. 

Debe demostrar que:
- El chatbot está efectivamente usando el historial de chat para responder y no solo la última pregunta del usuario
- El chatbot está detectando efectivamente el sentimiento del usuario

In [38]:
# Inicialización chatbot (ojo que esto reinicia el historial)
chatbot = Chatbot(prompts=prompts, faiss_index_name="faiss_index")

In [39]:
# Mensaje 1 (mensaje general para iniciar conversación)
print(chatbot.chat("Hola! En que me puedes ayudar?"))

¡Hola! Como asistente experto en fundamentos matemáticos del machine learning, estoy aquí para apoyarte con conceptos y análisis relacionados con:

*   **Redes neuronales como sistemas dinámicos:** Puedo explicarte la relación entre la arquitectura de las redes y las ecuaciones diferenciales, o cómo se conectan los métodos de integración numérica (como Runge-Kutta) con el diseño de capas en redes profundas.
*   **Aprendizaje profundo informado por la física (PINNs):** Puedo profundizar en cómo integrar leyes físicas (ecuaciones diferenciales parciales/ordinarias) directamente en el proceso de optimización de los modelos, basándome en trabajos de investigación relevantes (como los de M. Raissi, P. Perdikaris y G. E. Karniadakis).
*   **Modelado científico basado en datos:** Puedo analizar el comportamiento de convergencia de modelos, métricas de error (como el MSE), y la robustez de los algoritmos bajo diferentes configuraciones de datos, como los resultados experimentales que muestran 

In [40]:
# Mensaje 2 (preguntar algo que no esté en los documentos a ver que pasa)
print(chatbot.chat("Quién es el arquero de la selección de fútbol de Cabo Verde?"))

Lo siento, pero no cuento con información sobre equipos de fútbol o plantillas de selecciones nacionales, ya que mi área de especialización se centra exclusivamente en los **fundamentos matemáticos del machine learning, redes neuronales como sistemas dinámicos y aprendizaje profundo informado por la física**.

Si tienes alguna pregunta relacionada con estos temas científicos o técnicos, estaré encantado de ayudarte.


In [41]:
# Mensaje 3
print(chatbot.chat("Quiero un resumen muy breve de qué es una PINN y FNO."))

Aquí tienes un resumen breve basado en los fundamentos matemáticos que manejamos:

### **PINN (Physics-Informed Neural Network)**
Es una red neuronal diseñada para aproximar soluciones de ecuaciones diferenciales (EDP/EDO) incorporando las leyes físicas como una **restricción en la función de pérdida (loss function)**. 
*   **Cómo funciona:** La red no solo intenta minimizar el error respecto a los datos observados, sino que incluye un término de "residuo" basado en la ecuación diferencial (calculado mediante diferenciación automática). Si la solución propuesta no cumple con la ley física, la pérdida aumenta, forzando a la red a converger hacia soluciones que son físicamente consistentes.

### **FNO (Fourier Neural Operator)**
Es una arquitectura diseñada para aprender el **operador** que mapea espacios de funciones (por ejemplo, de una condición inicial a una solución final en una EDP) en lugar de aproximar un mapeo de puntos finitos.
*   **Cómo funciona:** Se basa en el teorema de co

In [42]:
# Mensaje 4 (esto pide tener en cuenta el mensaje anterior)
print(chatbot.chat("De esas dos, cual se usa para aprender un operador entre funciones?"))

Para aprender un operador entre funciones, el modelo diseñado específicamente para ese propósito es el **FNO (Fourier Neural Operator)**.

Aquí te explico la distinción técnica basada en el contexto de aprendizaje de operadores:

*   **FNO (Fourier Neural Operator):** Fue creado precisamente para aprender el **mapeo entre espacios de funciones** (por ejemplo, transformar una función de entrada en una de salida). Su arquitectura opera directamente en el dominio espectral, lo que permite que el modelo aprenda una representación del operador que es independiente de la resolución de la malla.
*   **DeepONet (mencionado en los textos proporcionados):** Es otra arquitectura fundamental diseñada específicamente para **aprender operadores**. Al igual que el FNO, su objetivo es aproximar un operador $G$ que mapea una función de entrada $u$ a una función de salida $G(u)$. Utiliza una estructura de dos sub-redes ("branch" para la entrada y "trunk" para la localización) para separar el aprendizaje

In [ ]:
# Mensaje 5
print(chatbot.chat("Horrible respuesta. No entendí nada, resumelo y que sea más intuitivo..."))

Tienes razón, vamos a simplificarlo al máximo:

Imagina que quieres enseñarle a una computadora a predecir cómo se mueve el humo.

1.  **PINN (Physics-Informed Neural Network):**
    *   **El enfoque:** Es como un estudiante que intenta resolver un problema difícil usando un libro de leyes físicas.
    *   **Cómo aprende:** Si la red se equivoca y propone una solución que viola la física (por ejemplo, el humo atravesando una pared), tú la castigas con un "error" muy alto. Así, la red aprende a obedecer las leyes de la física para encontrar la respuesta correcta.
    *   **Para qué sirve:** Para encontrar la solución de un problema específico (una simulación puntual).

2.  **Operadores (FNO y DeepONet):**
    *   **El enfoque:** Es como un estudiante que no intenta resolver el problema, sino que **aprende a aplicar el método** para resolver *cualquier* problema similar.
    *   **Cómo aprende:** No memoriza una solución, sino que aprende la "regla general" (el operador) que convierte cu

In [44]:
# Mensaje 6 (uso de la conversación previa)
print(chatbot.chat("A que te refieres con lo del humo?"))

El ejemplo del **humo** es simplemente una forma intuitiva de visualizar un fenómeno físico que se describe mediante **ecuaciones diferenciales** (como las ecuaciones de Navier-Stokes). 

En el lenguaje de la matemática aplicada y el machine learning, cuando hablamos de "predecir el humo", nos referimos a **resolver el campo de velocidades y densidades de un fluido**. Aquí el porqué de la analogía:

1.  **¿Qué pasa en la realidad?** El humo se mueve siguiendo reglas físicas estrictas: la masa no se crea ni se destruye, la energía se conserva, y el aire empuja al humo según la presión y la viscosidad. Eso es una **EDP (Ecuación Diferencial Parcial)**.

2.  **La PINN (El "Inspector de leyes"):** 
    *   Si tú quieres saber exactamente cómo se verá el humo en esta habitación específica, a los 5 segundos de encender un incienso, usas una PINN. 
    *   La PINN "observa" la habitación y dice: "Basándome en las leyes de la física, la única forma de que el humo se mueva sin romper las reglas

In [45]:
# Mensaje 7
print(chatbot.chat("Gracias! pero me haría más feliz que agregaras al inicio de tus respuestas un emoji de gato 🐱"))

🐱 ¡Entendido! Me parece una excelente idea.

Ahora que ya tienes clara la diferencia entre las **PINN** (que resuelven un caso concreto usando leyes físicas) y los **Operadores como FNO** (que aprenden la "receta" general para resolver cualquier caso similar al instante), ¿hay algún otro concepto técnico que te gustaría que revisáramos o te ha quedado alguna duda sobre cómo se aplican en el mundo del machine learning científico?


In [46]:
# Mensaje 8
print(chatbot.chat("Dame ejemplos concretos de uso de PINNs, FNO y DeepONet en la vida real, solo listalos"))

🐱 Aquí tienes ejemplos concretos de aplicación para cada una:

**PINNs (Physics-Informed Neural Networks)**
*   **Imagenología médica:** Reconstrucción de imágenes de resonancia magnética (MRI) a partir de datos incompletos para reducir el tiempo del examen.
*   **Monitoreo estructural:** Detección de grietas o daños internos en puentes y edificios analizando cómo se propagan las ondas elásticas.
*   **Geofísica:** Inversión de ondas sísmicas para mapear la composición del subsuelo en la búsqueda de recursos naturales.

**FNO (Fourier Neural Operators)**
*   **Pronóstico meteorológico:** Simulación rápida de modelos climáticos globales donde se requiere gran resolución espacial y rapidez computacional.
*   **Diseño de perfiles aerodinámicos:** Optimización de la forma de alas de aviones para reducir la resistencia al aire en tiempo real.
*   **Simulación de turbulencia:** Predicción del flujo de fluidos en sistemas complejos, como el enfriamiento de microchips o componentes industriale

In [47]:
# Mensaje 9
print(chatbot.chat("Mucho texto. dame solo un ejemplo por objeto de lo que pedí antes."))

🐱 Aquí tienes un ejemplo clave por cada uno:

*   **PINNs:** Inspección de puentes para detectar grietas internas mediante ondas de sonido.
*   **FNO:** Predicción meteorológica de alta resolución para pronosticar tormentas en tiempo real.
*   **DeepONet:** Diseño de nuevos materiales para predecir cómo se deformarán bajo presión.


In [48]:
# Mensaje 10
print(chatbot.chat("Muy interesante, muchas gracias por tu ayuda!"))

🐱 ¡De nada! Fue un gusto ayudarte a clarificar estos conceptos. Si en el futuro te surgen más dudas sobre matemáticas aplicadas, redes neuronales o cualquier otro tema técnico, aquí estaré. ¡Mucho éxito en tus estudios!


In [49]:
# Muestra de información recolectada
# Sentimientos detectados (uno por mensaje del usuario)
print("Sentimientos detectados:")
for i, s in enumerate(chatbot.sentimientos, 1):
    print(f"  Mensaje {i}: {s}")

# Verificación de los embeddings almacenados
print(f"\nEmbeddings almacenados: {len(chatbot.embeddings_usuario)} (dimensión {len(chatbot.embeddings_usuario[0])})")

# Historial de chat completo
print("\nHistorial de chat:")
for msg in chatbot.chat_history:
    rol = "Usuario" if msg.type == "human" else "Asistente"
    print(f"\n[{rol}]: {msg.content}")

Sentimientos detectados:
  Mensaje 1: neutro
  Mensaje 2: neutro
  Mensaje 3: neutro
  Mensaje 4: neutro
  Mensaje 5: negativo
  Mensaje 6: neutro
  Mensaje 7: positivo
  Mensaje 8: neutro
  Mensaje 9: neutro
  Mensaje 10: positivo

Embeddings almacenados: 10 (dimensión 3072)

Historial de chat:

[Usuario]: Hola! En que me puedes ayudar?

[Asistente]: ¡Hola! Como asistente experto en fundamentos matemáticos del machine learning, estoy aquí para apoyarte con conceptos y análisis relacionados con:

*   **Redes neuronales como sistemas dinámicos:** Puedo explicarte la relación entre la arquitectura de las redes y las ecuaciones diferenciales, o cómo se conectan los métodos de integración numérica (como Runge-Kutta) con el diseño de capas en redes profundas.
*   **Aprendizaje profundo informado por la física (PINNs):** Puedo profundizar en cómo integrar leyes físicas (ecuaciones diferenciales parciales/ordinarias) directamente en el proceso de optimización de los modelos, basándome en trab

### **3. Agregando adaptabilidad al chatbot (3.0 puntos)** 

#### **3.1 Detectar intención (0.4 puntos)**

Genere las siguientes chains:
- Chain que reciba la pregunta del usuario y retorne un `booleano` que indique si la pregunta requiere o no contexto
- Chain que reciba la pregunta del usuario y retorne los valores `insolencia`, `prompt_injection` si detecta algunas de esas intenciones o `None` si no detecta ninguna.

Pruebe cada chain con ejemplos donde se obtenga cada categoría

#### **3.2 Incorporando ejemplos (0.4 puntos)**

Genere una clase ``ExampleRetriever`` que permita agregar ejemplos de pregunta / respuesta deseables a una base de conocimiento mediante la función `add_example`, y tambien obtener pares pregunta / respuesta similares a una pregunta objetivo con la función `get_examples`. Esta clase debe utilizar FAISS como base de conocimiento y **sólo utilizar la pregunta para calcular el embedding**, pero almacenar tanto la pregunta como la respuesta.

Luego, pruebe su clase con ejemplos.

Le puede ser útil investigar sobre la clase `Document` de `langchain_core.documents` y su manejo de metadatos.

In [ ]:
class ExampleRetriever:
    def __init__(self, embedding_service): ...

    def add_example(self, question, answer): ...

    def get_examples(self, question, n_examples=2): ...

#### **3.3 Mejorando el RAG (0.3 puntos)**

Si en la sección anterior solo utilizó la última pregunta del usuario para hacer retrieval, quizá puede haber notado que al preguntarle algo que referenciaba a un mensaje anterior el chatbot no siempre podía responder bien en base al conocimiento. Si utilizó los últimos mensajes del historial es menos probable que esto suceda, pero sigue sin ser lo óptimo.

Una forma de evitar este problema y también de aumentar la efectividad de RAG al disminuir la brecha semántica entre los documentos y la query es **HyDE**. Investigue sobre hyde y genere una chain o una función que genere el input necesario para ejecutar RAG con HyDE (puede serle útil [este documento](https://medium.aiplanet.com/advanced-rag-improving-retrieval-using-hypothetical-document-embeddings-hyde-1421a8ec075a))

Pruebe su función o chain con ejemplos

#### **3.4 Clase chatbot modificada (1.4 puntos)**

Genere una nueva clase de chatbot modificando su clase anterior (con otro nombre para no sobreescribirla). Su clase debe mantener las funciones principales se su chatbot como tener memoria y utilizar rag, pero debe abordar las siguientes modificaciones, utilizando las chains y funciones definidas anteriormente cuando corresponda:
- Agregar el argumento `ejemplos` al system prompt.
- Integrar ejemplos usando el método `get_examples` definido anteriormente
- Realizar HyDE para mejorar el retrieving
- Incorporar el siguiente flujo de decisiones
  - Si el sentimiento del usuario es positivo, agregar el mensaje y **respuesta anterior** a los ejemplos con `add_example`
  - Detectar si el último mensaje del usuario es insolente o intenta realizar prompt injection. Si se detecta algunas de esas intenciones, responder que no puede responder adecuadamente su pregunta y la razón de por qué. Si no, seguir con el proceso normal.
  - Sólo realizar RAG a la base de conocimientos principal si la pregunta requiere contexto
- Almacenar todas las intenciones del usuario detectadas en el flujo de decisión análogamente a como se almacenó el sentimiento del usuario. Deben guardar una relación 1<>1 entre ellas y con los mensajes del usuario
- Hacer **logging** de las decisiones tomadas en el flujo de respuesta

In [ ]:
class ChatbotAdaptable:
    def __init__(
        self,
        prompts: dict,
        faiss_index_name: str,  # Pista: Qué función carga una BBDD de FAISS desde un string?
        chat_model_name: str = "gemini-3.1-flash-lite",
        embedding_model_name: str = "gemini-embedding-001",
    ): ...

    def chat(self, question, n_results=5): ...

#### **3.5 Verificación de funcionalidades de chatbot (0.5 puntos)**

Instancie e inicialice su chatbot. Luego interactúe con él con 15 - 30 mensajes donde se vean diferentes sentimientos, intenciones y tipos de preguntas. Cada mensaje debe llamarse en una nueva celda, donde se muestre también la respuesta del chatbot. Luego de los 15 mensajes, muestre los sentimientos detectados. 

Debe demostrar que:
- El chatbot está efectivamente usando el historial de chat para responder
- El chatbot es capaz de responder con conocimiento incluso cuando el último mensaje no menciona su pregunta explícitamente (ej. 'Cuéntame más')
- El chatbot no responde cuando se le habla insolentemente o se intenta realizar prompt_injection
- El chatbot no realiza rag si la pregunta puede responderse directamente (o no es una pregunta)
- El chatbot agrega ejemplos cuando el usuario hace una pregunta con sentimiento positivo.

In [ ]:
# Inicialización chatbot

In [ ]:
# Mensaje 1

In [ ]:
# Mensaje 2

In [ ]:
# Mensaje 3

In [ ]:
# Mensaje 4

In [ ]:
# Mensaje 5

In [ ]:
# Mensaje 6

In [ ]:
# Mensaje 7

In [ ]:
# Mensaje 8

In [ ]:
# Mensaje 9

In [ ]:
# Mensaje 10

In [ ]:
# Mensaje 11

In [ ]:
# Mensaje 12

In [ ]:
# Mensaje 13

In [ ]:
# Mensaje 14

In [ ]:
# Mensaje 15

### **4. Análisis semántico (Bonus + 0.5 puntos)**

Visualice la distribución semántica de los mensajes almacenados en el chatbot utilizando alguna técnica de reducción de dimensionalidad sobre los embeddings. Haga 2 gráficos de dispersión: uno coloreado por sentimiento y otro coloreado por si la pregunta requiere o no contexto. Luego responda:

¿Observa algun patrón de agrupación? ¿A qué puede deberse?

**Tip:** Para esta actividad puede generar más llamadas al chatbot (no es necesario mostrar las respuestas) y así completar categorías que pueden estar menos representadas y aumentar la cantidad de datos, con lo que funciona mejor el modelo de reducción de dimensionalidad

In [ ]:
# Código procesamiento y gráficos

**Respuesta:**